In [1]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.7/567.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 695.9 kB/s eta 0:00:00


In [2]:
from typing import List
import torch, copy, random, os, json
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.benchmarks import IFEval


device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH  = "/kaggle/input/models/faihaj/lfm-230m/transformers/default/1"
RESULTS_CSV = "/kaggle/working/clean_results.csv"

INITIAL_SEED = 60
N_SEEDS    = 20
N_PROBLEMS = 100

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, dtype=torch.bfloat16
).to(device)
model.eval()




# ── LFM2 wrapper ──────────────────────────────────────────────────────────────
class LFM2(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.model = model; self.tokenizer = tokenizer
    def load_model(self): return self.model
    def generate(self, prompt: str) -> str:
        inputs = self.tokenizer([prompt], return_tensors="pt").to(device)
        try:
            ids = self.model.generate(
                **inputs, max_new_tokens=100,
                do_sample=False, temperature=None, top_p=None
            )
            return self.tokenizer.batch_decode(
                ids, skip_special_tokens=True
            )[0]
        except RuntimeError:
            return ""
    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)
    def get_model_name(self): return "LFM2-230M"
    def __call__(self, prompt: str) -> str: return self.generate(prompt)

# ══════════════════════════════════════════════════════════════════════════════
# MAIN EXPERIMENT — every method now evaluated under every fault type
# ══════════════════════════════════════════════════════════════════════════════
rows = []
for seed in range(INITIAL_SEED, N_SEEDS + INITIAL_SEED):
    random.seed(seed); torch.manual_seed(seed)
    lfm   = LFM2(model=model, tokenizer=tokenizer)
    # Pick a random subset of tasks
    bench = IFEval(n_problems=N_PROBLEMS)
    bench.evaluate(model=lfm)
    rows.append({"seed": seed, "accuracy": bench.overall_score})

pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)

Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

ifeval_input_data.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/541 [00:00<?, ? examples/s]

Processing 100 IFEval problems: 100%|██████████| 100/100 [06:22<00:00,  3.83s/it]


...

Overall IFEval Accuracy: 0.6800
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.8571
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.6667
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc

Processing 100 IFEval problems: 100%|██████████| 100/100 [06:14<00:00,  3.75s/it]


Overall IFEval Accuracy: 0.6800
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.8571
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.6667
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc

Processing 100 IFEval problems: 100%|██████████| 100/100 [06:42<00:00,  4.03s/it]

Overall IFEval Accuracy: 0.6800
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.8571
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.6667
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc